In [ ]:
import os
import warnings # 避免一些可以忽略的报错
warnings.filterwarnings('ignore')
import random
import gc
import copy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm # 进度条
import time

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

In [ ]:
is_debug = False

class CONFIG:
    seed = 308

    test_batch_size = 512
    
    in_features = 784
    n_classes = 10

    n_workers = 1
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    test_csv = "/kaggle/input/digit-recognizer/test.csv"
    ckpt_path = "/kaggle/input/notebook-xunlian/BestModel.pth"

In [ ]:
def set_seed(seed=308):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    
set_seed(CONFIG.seed) # 固定随机种子，方便结果复现

In [ ]:
test = pd.read_csv(CONFIG.test_csv) # 读取 测试集 的数据
test

In [ ]:
class MyDataset(Dataset):
    def __init__(self, df):
        super().__init__()
        self.df = df
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx, :] # 从 df 中取出 idx 这一行
        X = row["pixel0":].values
        
        return X, str(idx) # 测试集的数据没有 label，我们用它对应的索引代替

In [ ]:
def prepare_loaders():
    test_datasets = MyDataset(df=test)
    
    test_loader = DataLoader(test_datasets, batch_size=CONFIG.test_batch_size, num_workers=CONFIG.n_workers, shuffle=False, pin_memory=True)
    # 推理过程对数据按顺序执行，不进行打乱操作，shuffle为False
    
    return test_loader

In [ ]:
class DigitRecognizerModel(nn.Module):
    def __init__(self):
        super(DigitRecognizerModel, self).__init__()

        self.l1 = nn.Sequential(
            nn.Linear(28*28, 2048, bias=True), 
            nn.BatchNorm1d(2048),
            nn.ReLU(inplace=True), 
            # nn.Dropout()      如果使用dropout那么epoch要加大一些
        )

        self.l2 = nn.Sequential(
            nn.Linear(2048, 512, bias=True),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            # nn.Dropout() 
        )
        
        self.l3 = nn.Sequential(
            nn.Linear(512, 10, bias=True),
            nn.BatchNorm1d(10),
            nn.ReLU(inplace=True), 
            # nn.Dropout() 
        )
    
    def forward(self, x):
        x = x.view(-1, 28*28)
        
        x = self.l1(x)
        x = self.l2(x)
        x = self.l3(x)
        
        return x 

In [ ]:
model = DigitRecognizerModel() # 实例化模型
model.to(CONFIG.device)

# 加载训练得到的模型的权重参数
model.load_state_dict(torch.load(CONFIG.ckpt_path, map_location=CONFIG.device)) # map_location作用：防止不同device之间的模型与参数无法加载

In [ ]:
def Infer(model, test_loader):
    model.eval()
    
    y_preds = []
    bar = tqdm(enumerate(test_loader), total=len(test_loader))
    with torch.no_grad():
        for step, (X, X_id) in bar:

            X = X.to(CONFIG.device, dtype=torch.float)

            outputs = model(X)
            outputs = F.softmax(outputs)

            y_preds.append(outputs.argmax(1).detach().cpu().numpy())
            
    y_preds = np.concatenate(y_preds)

    return y_preds

In [ ]:
test_loader = prepare_loaders()

In [ ]:
y_preds = Infer(model, test_loader)

In [ ]:
sub = pd.DataFrame()
sub["ImageId"] = test.index + 1
sub["Label"] = y_preds
sub


In [ ]:
sub.to_csv('submission.csv', index=False)
pd.read_csv('submission.csv')